# Phase 6 — PoT Activation Quantization

**Base**: Phase 4 (SparseShift + PoT-BN)
**Change**: Add `PoTActivation` after every ReLU in the residual stack

Post-ReLU activations → `{0} ∪ {2^p : p_min ≤ p ≤ p_max}`
- p_max = floor(log2(α)), where α is a **learnable** per-layer clip parameter
- n_levels = 8 non-zero PoT values (default), α_init = 4.0
- Warmup: PoT activation activates after epoch 30 (same as PoT-BN)

With PoT weights + PoT-BN scales + PoT activations, every multiply in the
conv stack is a bit-shift → **fully multiply-free** forward pass end-to-end.

Models:
- `sparseshift_fixed50_potbn_w30_act` — fixed 50% sparsity
- `sparseshift_learnable_potbn_w30_act` — learnable sparsity

## Setup — clone repo and install dependencies

In [ ]:
import os, subprocess
if not os.path.exists('/kaggle/working/OmniShift'):
    subprocess.run(['git','clone','https://github.com/CryAndRRich/OmniShift.git','/kaggle/working/OmniShift'], check=True)
os.chdir('/kaggle/working/OmniShift')
!pip install -q pyyaml

## Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU only'
print(f'Device : {gpu_name}')
print(f'PyTorch: {torch.__version__}')

## Training runs

### CIFAR-10 · fixed50 + PoT-BN + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase6_pot_act.yaml --name sparseshift_fixed50_potbn_w30_act --dataset cifar10

### CIFAR-10 · learnable + PoT-BN + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase6_pot_act.yaml --name sparseshift_learnable_potbn_w30_act --dataset cifar10

### SVHN · fixed50 + PoT-BN + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase6_pot_act.yaml --name sparseshift_fixed50_potbn_w30_act --dataset svhn

### SVHN · learnable + PoT-BN + PoT activations

In [ ]:
!python scripts/run_experiment.py --config configs/phase6_pot_act.yaml --name sparseshift_learnable_potbn_w30_act --dataset svhn

## Results summary

In [ ]:
!python scripts/summarize_results.py